# 02 - Raw Data Quality Assessment

Scans the RAW layer (read-only) and reports, per dataset:
row/column counts, missing cells, duplicate rows, null/duplicate
primary keys, invalid numbers and dates, plus logical checks
(order-total consistency, inventory balance, rating range,
discount range, price/cost margin).

This is the **pre-cleaning** view: the non-zero values you see
below are the documented injected quality cases.


In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import nb_common

# ------------------------------------------------------------------
# Run mode:
#   "quick" -> demo-scale data (~20k orders) in notebook/outputs/
#   "full"  -> full SRS scale (1M lines) in the canonical repo dirs
# ------------------------------------------------------------------
MODE = "quick"

paths = nb_common.setup_paths(MODE)
nb_common.ensure_config(paths)
mk = nb_common.markers(paths)

pd.options.display.max_columns = 20
print(f"MODE: {MODE}")
print(f"raw       : {paths['raw']}")
print(f"processed : {paths['processed']}")


MODE: quick
raw       : /home/user/Techwizz/notebook/outputs/quick/raw
processed : /home/user/Techwizz/notebook/outputs/quick/processed


In [2]:
if not nb_common.step_done(*mk['generation']):
    from generate_dineiq_data import generate
    generate(raw_dir=paths['raw'], config_path=paths['config'])

In [3]:
import data_quality_check

if not nb_common.step_done(*mk['quality']):
    report, summary = data_quality_check.main(raw_dir=paths['raw'], report_dir=paths['reports'])
else:
    print('Quality report already exists - skipping (delete data_quality_report.csv to rerun).')
    report = pd.read_csv(paths['reports'] / 'data_quality_report.csv', low_memory=False)
    summary = pd.read_csv(paths['reports'] / 'data_quality_summary.csv', low_memory=False)

DineIQ Analytics - Raw Data Quality Assessment
Raw data directory: /home/user/Techwizz/notebook/outputs/quick/raw

Checking: locations.csv
  Rows: 20 | Columns: 4 | Missing: 0 | Duplicates: 0
Checking: restaurants.csv
  Rows: 20 | Columns: 5 | Missing: 0 | Duplicates: 0
Checking: menu_categories.csv
  Rows: 10 | Columns: 2 | Missing: 0 | Duplicates: 0
Checking: menu_items.csv
  Rows: 150 | Columns: 9 | Missing: 0 | Duplicates: 0
Checking: customers.csv
  Rows: 8,040 | Columns: 11 | Missing: 80 | Duplicates: 40
Checking: orders.csv
  Rows: 20,000 | Columns: 14 | Missing: 17,317 | Duplicates: 0
Checking: order_items.csv


  Rows: 200,040 | Columns: 7 | Missing: 0 | Duplicates: 0
Checking: pricing_history.csv
  Rows: 272 | Columns: 7 | Missing: 0 | Duplicates: 0
Checking: promotions.csv
  Rows: 120 | Columns: 11 | Missing: 0 | Duplicates: 0
Checking: ratings.csv
  Rows: 20,000 | Columns: 8 | Missing: 0 | Duplicates: 0
Checking: inventory.csv
  Rows: 10,000 | Columns: 10 | Missing: 0 | Duplicates: 0
Checking: wastage.csv
  Rows: 10,000 | Columns: 7 | Missing: 0 | Duplicates: 0

QUALITY ASSESSMENT COMPLETE
Detailed report: /home/user/Techwizz/notebook/outputs/quick/reports/data_quality_report.csv
Summary report : /home/user/Techwizz/notebook/outputs/quick/reports/data_quality_summary.csv

            dataset   rows  missing_cells  duplicate_rows
      locations.csv     20              0               0
    restaurants.csv     20              0               0
menu_categories.csv     10              0               0
     menu_items.csv    150              0               0
      customers.csv   8040       

In [4]:
print('--- Per-dataset summary ---')
summary

--- Per-dataset summary ---


,dataset,rows,missing_cells,duplicate_rows
0,locations.csv,20,0,0
1,restaurants.csv,20,0,0
2,menu_categories.csv,10,0,0
3,menu_items.csv,150,0,0
4,customers.csv,8040,80,40
5,orders.csv,20000,17317,0
6,order_items.csv,200040,0,0
7,pricing_history.csv,272,0,0
8,promotions.csv,120,0,0
9,ratings.csv,20000,0,0


In [5]:
# Non-zero issue rows from the detailed report (the injected quality cases)
issues = report[report['value'] != 0].copy()
issues = issues[~issues['check'].isin(['ROWS', 'COLUMNS'])]
issues

,dataset,check,value,details
35,customers.csv,MISSING_CELLS,80,
36,customers.csv,DUPLICATE_ROWS,40,
38,customers.csv,DUPLICATE_PRIMARY_KEY,40,customer_id
43,orders.csv,MISSING_CELLS,17317,
72,order_items.csv,NEGATIVE_line_total,183,line_total
73,order_items.csv,NON_POSITIVE_QUANTITY,217,Order item quantity must be greater than zero


**Reading the output:** every non-zero row above is expected - the
generator injects these cases at the rates in
`Ali Jaan/config/data_generation_config.yaml`. The cleaning notebook
next shows them being detected, quarantined, and logged.
